In [73]:
import random
import math
import numpy as np
import copy

def print_matrix(M):
    for row in M:
        row_rounded = [round(x, 2) for x in row]
        row_no_negative_zeros = [0.0 if math.isclose(x, 0.0) else x for x in row_rounded]
        print(" ".join(f"{x:.2f}" for x in row_no_negative_zeros))
    print()


def divide_row(row, k, M):
    n = len(M)
    for j in range(n):
        M[row][j] /= k


def swap_rows(row1, row2, M):
    n = len(M)
    for j in range(n):
        M[row1][j], M[row2][j] = M[row2][j], M[row1][j]


def add_scaled_row_to_another(row1, row2, k, M):
    n = len(M)
    for j in range(n):
        M[row2][j] += M[row1][j] * k


def gauss_elimination(M, pivoting = True, normalize = True):
    n = len(M)

    for j in range(n): 
        if pivoting:
            maxi, pivot_idx = 0.0, 0
            for i in range(j, n):
                if abs(M[i][j]) > maxi :
                    maxi, pivot_idx = abs(M[i][j]), i
            if pivot_idx != j: # we need to swap
                swap_rows(j, pivot_idx, M)
        if normalize:
            divide_row(j, M[j][j], M) # we want our row to have 1.0 as the element on the diagonal

        for k in range(j + 1, n):
            assert not np.isclose(M[j][j], 0)
            add_scaled_row_to_another(j, k, -M[k][j]/M[j][j], M) 


def inverse_matrix(M):
    n = len(M)
    M_res = [[0 for _ in range(n)] for _ in range(n)]
    for i in range(n):
        M_res[i][i] = 1
    
    for j in range(n):
        maxi, pivot_idx = 0.0, 0 
        for i in range(j, n):
            if abs(M[i][j]) > maxi :
                maxi, pivot_idx = abs(M[i][j]), i 

        if pivot_idx != j:
            swap_rows(j, pivot_idx, M)
            swap_rows(j, pivot_idx, M_res)

        div = M[j][j]
        divide_row(j, div, M) 
        divide_row(j, div, M_res)

        for k in range(n):
            if k != j:
                scalar = -M[k][j]
                add_scaled_row_to_another(j, k, scalar, M) 
                add_scaled_row_to_another(j, k, scalar, M_res)

    return M_res



n = 6
M1 = [[random.randint(0, 15) for _ in range(n)] for _ in range(n)]
M2 = copy.deepcopy(M1)


print()
print_matrix(M1)
print("\nAFTER GAUSS ELIMINATION:\n")
gauss_elimination(M1)
print_matrix(M1)


if np.linalg.det(np.array(M2)) != 0:
    print("\nINVERSE MATRIX\n")
    print_matrix(inverse_matrix(M2))


3.00 5.00 4.00 6.00 11.00 12.00
9.00 3.00 14.00 1.00 13.00 12.00
14.00 10.00 6.00 1.00 5.00 1.00
0.00 3.00 5.00 3.00 7.00 12.00
11.00 7.00 7.00 12.00 9.00 4.00
15.00 0.00 9.00 9.00 13.00 13.00


AFTER GAUSS ELIMINATION:

1.00 0.00 0.60 0.60 0.87 0.87
0.00 1.00 -0.24 -0.74 -0.71 -1.11
0.00 0.00 1.00 -0.23 0.79 0.81
0.00 0.00 0.00 1.00 0.25 0.05
0.00 0.00 0.00 0.00 1.00 1.66
0.00 0.00 0.00 0.00 0.00 1.00


INVERSE MATRIX

-0.03 -0.04 0.06 0.00 -0.04 0.08
0.02 -0.02 0.07 0.06 0.02 -0.06
-0.17 0.08 -0.04 0.11 0.09 -0.05
-0.06 -0.04 -0.05 0.07 0.10 0.01
0.27 0.10 -0.05 -0.31 -0.03 -0.04
-0.08 -0.07 0.04 0.18 -0.05 0.06



In [107]:
def gauss_numpy(A, pivoting = True, normalize = True):
    n = len(A)
    A = np.array(A, dtype=np.float32)
    for i in range(n):
        if pivoting or np.isclose(A[i, i], 0.0):
            pivot_idx = i + np.argmax(np.abs(A[i:, i])) if pivoting else i + np.where(~np.isclose(np.abs(A[i:, i]), 0.0))[0][0]
            if pivot_idx != i:
                A[[i, pivot_idx]] = A[[pivot_idx, i]]
        if normalize:
            A[i, :] /= A[i, i]
        for j in range(i+1, n):
            factor = A[j][i]/A[i, i]
            A[j, :] -= factor * A[i, :]
    return A

def lu_factorization(A, pivoting = True):
    n = len(A)
    P = [i for i in range(n)]
    A = np.array(A, dtype=np.float32)
    L, U = np.array([[0.0 for _ in range(n)] for _ in range(n)]), np.copy(A)
    for i in range(n):
        L[i, i] = 1.0
    for i in range(n):
        
        if pivoting or np.isclose(U[i, i], 0.0):
            pivot_idx = i + np.argmax(np.abs(U[i:, i])) if pivoting else i + np.where(~np.isclose(np.abs(A[i:, i]), 0.0))[0][0]
            if not pivoting and np.isclose(U[i, i], 0.0):
                assert i != pivot_idx
                print("Found zero", U[pivot_idx, i])
                print(np.abs(U[i:, i]))
        
            if pivot_idx != i:
                U[[i, pivot_idx]] = U[[pivot_idx, i]]
                L[[i, pivot_idx], :i] = L[[pivot_idx, i], :i]
                P[i], P[pivot_idx] = P[pivot_idx], P[i]

        for j in range(i+1, n):
            factor = U[j][i]/U[i, i]
            U[j, :] -= factor * U[i, :]
            L[j, i] = factor

    P = np.array(P)
    P_res = np.zeros((P.size, P.max() + 1))
    P_res[np.arange(P.size), P] = 1
    return P_res, L , U

NUM_TRIALS = 100
n = 10

for i in range(NUM_TRIALS):
    M3 = [[random.randint(0, 15) for _ in range(n)] for _ in range(n)]
    M3 = np.array(M3)
    P2, L2, U2 = lu_factorization(M3)
    P3, L3, U3 = lu_factorization(M3, False)

    # print(np.round(L2, 2))
    # print()
    # print(np.round(U2, 2))
    # print()
    # print(np.round(np.array(M3), 2))
    # print()
    # print(np.round(np.transpose(P3) @ L3 @ U3, 2))
    # print()
    # print(np.allclose(P2 @ M3, L2 @ U2))

    pivot, no_pivot = np.max(P2 @ M3 - L2 @ U2), np.max(P3 @ M3 - L3 @ U3)
    # print("With pivoting: ", pivot)
    # print("Without pivoting: ", no_pivot)
    # print("Diff", round(no_pivot/pivot, 2))

M4 = [[random.randint(0, 15) for _ in range(n)] for _ in range(n)]
print(np.round(gauss_numpy(M4), 2))






Found zero 9.0
[ 0.  9.  8.  9.  0. 10.  9. 12. 13.  7.]
Found zero 12.0
[ 0. 12.  4. 14. 14.  4. 11.  9. 13.  7.]
Found zero 9.0
[ 0.  9.  2.  4.  7.  2.  5. 11. 14.  2.]
Found zero 4.0
[ 0.  4.  0. 14.  4. 11.  4.  6.  4.  2.]
Found zero -11.0
[ 0.  11.  16.   5.   3.   8.5 11.   0.5  0. ]
Found zero 9.0
[ 0.  9.  4. 11.  6. 10. 14. 10.  6.  6.]
Found zero 7.0
[ 0.  7.  2. 12.  7.  3.  0. 11.  2.  2.]
Found zero 4.0
[ 0.  4.  6.  1. 14. 15. 10.  4.  2.  1.]


AssertionError: 